   event_id    event_name      lat      lon
0         1       Пожар_1  55.7558  37.6173
1         2  Наводнение_1  48.8566   2.3522
2         3     Вырубка_1 -14.2350 -51.9253
3         4       Шторм_1   0.0000   0.0000


/home/nvmaxim/Projects/dev-sandbox/.venv/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: File dljffa has GPKG application_id, but non conformant file extension
  ogr_write(


In [21]:
import numpy as np
from osgeo import gdal
import matplotlib.pyplot as plt

# --- 1. Подготовка: создаем тестовый файл slope.tif на диске ---
rows, cols = 200, 200
# Генерируем случайную матрицу высот от 0 до 500 метров
fake_slope = np.random.randint(0, 500, size=(rows, cols)).astype(np.float32)

# Сохраняем её как GeoTIFF через GDAL
driver = gdal.GetDriverByName('GTiff')
out_ds = driver.Create('slope.tif', cols, rows, 1, gdal.GDT_Float32)
out_ds.GetRasterBand(1).WriteArray(fake_slope)
out_ds.FlushCache()  # Записываем на диск
out_ds = None        # Закрываем файл, чтобы освободить ресурсы


# --- 2. РЕШЕНИЕ УПРАЖНЕНИЯ 3 ---

# Шаг А: Ленивое открытие растра
ds = gdal.Open('slope.tif')
band = ds.GetRasterBand(1)

# Шаг Б: Загрузка пикселей в RAM в виде NumPy массива
array = band.ReadAsArray()

# Шаг В: Вычисление среднего и векторизованная маска
mean_val = np.mean(array)
binmask = np.where(array >= mean_val, 1, 0)

# Закрываем датасет
ds = None


# --- 3. Проверка результата в консоли ---
print(f"Размер растровой матрицы: {array.shape}")
print(f"Среднее значение высоты: {mean_val:.2f}")
print(f"Уникальные значения в бинарной маске: {np.unique(binmask)}")

Размер растровой матрицы: (200, 200)
Среднее значение высоты: 248.08
Уникальные значения в бинарной маске: [0 1]


/home/nvmaxim/Projects/dev-sandbox/.venv/lib/python3.12/site-packages/osgeo/gdal.py:606: FutureWarning: Neither gdal.UseExceptions() nor gdal.DontUseExceptions() has been explicitly called. In GDAL 4.0, exceptions will be enabled by default.
  warnings.warn(


Для упражнения со spatial join (sjoin) там же есть встроенный датасет naturalearth_cities (города как точки).

2. Генерация "на лету" (OSMnx и GEE)
В Упражнениях 5 (OSMnx) и 6 (Earth Engine) тебе вообще не нужны локальные файлы. Код из этих упражнений сам скачивает данные по API прямо в оперативную память. Тебе нужно только наличие интернета.

3. Создание фиктивных данных (Синтетика)
Иногда проще создать два квадрата руками через Shapely, чтобы протестировать, как работает sjoin или расчет площади:

In [6]:
import geopandas as gpd
from shapely.geometry import Polygon

# Создаем простейший квадрат 10x10
poly1 = Polygon([(0, 0), (10, 0), (10, 10), (0, 10)])
gdf = gpd.GeoDataFrame(geometry=[poly1], crs="EPSG:4326")

print(poly1)
print(gdf)

POLYGON ((0 0, 10 0, 10 10, 0 10, 0 0))
                                  geometry
0  POLYGON ((0 0, 10 0, 10 10, 0 10, 0 0))


не тратим время на поиск реальных гигабайтных шейпфайлов. Для отработки навыков в dev-sandbox использовать gpd.datasets.get_path(...)